# G02: Run

*Does knowledge-graph evidence change where an LLM ranks the true post-cutoff drug?*

| Item | Definition | Quantification | Scoring |
|---|---|---|---|
| **Query** | one disease, a pool of 8 candidate drugs (1 positive, 7 distractors), candidates shown as `Drug-1` to `Drug-8` | the rank the positive receives; reciprocal rank; hit@1, @3, @5 | aggregated in G03 |
| **Arms** | `none`: no evidence; one arm per KG: each candidate carries that graph's dossier (targets, pathways; never the drug-disease edge) and the disease a profile (associated genes, pathways) | same queries, same seeds, same shuffles in every arm | the lift over `none` is the graph's contribution |
| **Repeats** | 3 seeds (distractor draws) x 3 shuffles (candidate orders) per query and arm | 9 calls per query per arm per model | averaged per query in G03 |

Every row of the run files carries `pair_id`, `family` and `arm`, so G03 pairs arms on the pair itself; nothing about query order has to be replayed.

**Reported, not scored.** The per-model summary at the end is a sanity check on the run; the paired analysis with intervals is G03.

**Modes.** `MODE = 'mock'` exercises pools, prompts, shuffles, parsing and scoring with a mock model and no graph loaded. `MODE = 'real'` calls the configured models and builds dossiers from the graphs; hosted models read their key from the environment or `.env`.

**Inputs:** `config/kgs.yaml` · `config/params.yaml` (`grounding`) · `results/checkpoints/G01_benchmark.pkl` · KG edge and node files (real mode) · `data/gold_standards/bridges/`

**Outputs:** `results/tables/G02_run/runs/<family>__<model>__seed<k>.csv` · `results/tables/G02_run/worked_examples/` · `results/checkpoints/G02_run.pkl`

Design notes and justifications: [`README.md`](README.md)

## Set-up

Reads the G01 checkpoint for the benchmark and rebuilds the candidate pools from the seeds. In real mode each KG is loaded once and its dossier index kept for every model and seed; in mock mode no graph is touched.

In [1]:
# Pick up edits to src/ without restarting the kernel. Without this, editing a
# module after it has been imported leaves the kernel holding the old version,
# which surfaces as a confusing ImportError for a name that is plainly on disk.
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass                      # not running under IPython (headless runner, plain python)

import gc
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Locate the repo root from any depth, then make the package importable.
# (`pip install -e .` also works and makes this block a no-op.)
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'config' / 'kgs.yaml').exists())
if str(_root / 'src') not in sys.path:
    sys.path.insert(0, str(_root / 'src'))

from IPython.display import display
import numpy as np
import pandas as pd


from tqdm.auto import tqdm

from biokgsuite.grounding import (KGDossiers, api_key_for, build_queries, is_api_model, load_distractor_pool,
                                  load_dotenv, load_entrez_to_symbol, mondo_children, ollama_models_available,
                                  run_family, run_path)
from biokgsuite.io import Outputs, find_config, load_config, require
from biokgsuite.viz.style import setup_style

setup_style()

In [2]:
# Config, parameters and the results writer.
config = load_config(find_config(_root), trial_subset=False)   # graphs are named below, not by the panel trial file
params = config['analysis_params']
G      = require(params, 'grounding')
BASE   = config['_base_dir']

# Every figure, table and checkpoint goes through `out`, as in notebooks 01-07.
out = Outputs('G02_run', BASE)

KGS       = require(G, 'kgs')
MODELS    = require(G, 'models')
LABELS    = require(G, 'model_labels') | {'none': 'No KG', 'primekg': 'PrimeKG', 'drkg': 'DRKG',
                                           'biokg': 'BioKG', 'openbilink': 'OpenBioLink', 'hetionet': 'Hetionet'}
FAMILIES  = require(G, 'families')
POOL_SIZE = require(G, 'pool_size')
SHUFFLES  = require(G, 'shuffles')
SEEDS     = require(G, 'seeds')
STRICT    = require(G, 'strict')
N_BOOT    = require(G, 'n_bootstrap')
SEED      = require(params, 'random_seed')
RNG       = np.random.default_rng(SEED)
# Run switches. Environment overrides let one launch run one model or one seed
# (for example on a cluster job) without editing the notebook. Real is the
# default; BKG_MODE=mock rehearses the pipeline with a deterministic stand-in
# and writes to its own folder, so a rehearsal can never be read as a result.
MODE     = os.environ.get('BKG_MODE', 'real')            # 'real' or 'mock'
if MODE not in ('real', 'mock'):
    raise ValueError(f'BKG_MODE must be real or mock, not {MODE!r}')
MOCK     = MODE == 'mock'
RUN_DIR  = Path(BASE) / 'results' / 'tables' / 'G02_run' / ('runs_mock' if MOCK else 'runs')
# BKG_LIMIT=n runs real calls on only the first n queries of each family: a
# smoke test of keys, dossiers and prompts for a few cents. Its files go to
# runs_smoke/, never runs/, because a truncated family cannot be analysed as
# a paired design and must not be mistaken for one.
LIMIT    = int(os.environ.get('BKG_LIMIT', '0'))
if LIMIT and not MOCK:
    RUN_DIR = RUN_DIR.parent / 'runs_smoke'
FORCE    = os.environ.get('BKG_FORCE', '0') == '1'      # recompute files that already exist
if os.environ.get('BKG_MODELS'):
    MODELS = [m.strip() for m in os.environ['BKG_MODELS'].replace('+', ',').split(',') if m.strip()]
elif MOCK:
    # A rehearsal costs nothing, so it also covers the Llama capacity ladder
    # that G04 reads; in real mode the ladder is launched per rung by
    # hpc/run_llm_ladder.sbatch with BKG_MODELS.
    MODELS = MODELS + [m for m in require(G, 'capacity_models') if m not in MODELS]
if os.environ.get('BKG_SEEDS'):
    SEEDS = [int(s) for s in os.environ['BKG_SEEDS'].split(',') if s.strip()]
if os.environ.get('BKG_FAMILIES'):
    FAMILIES = [f.strip() for f in os.environ['BKG_FAMILIES'].split(',') if f.strip()]
CAP             = require(G, 'cap')
ANONYMIZE       = require(G, 'anonymize')
ANONYMIZE_GENES = require(G, 'anonymize_genes')
BRIDGE_MODE     = require(G, 'bridge_mode')
TEMPERATURE     = require(G, 'temperature')
API_DELAY       = require(G, 'api_delay')
# A smoke test or a rehearsal keeps its worked examples beside its run files,
# so results/tables/G02_run/worked_examples/ only ever holds real prompts.
EXAMPLE_DIR     = (RUN_DIR / 'worked_examples') if (LIMIT or MOCK) else (out.tables / 'worked_examples')
load_dotenv(BASE)
if LIMIT:
    print(f'BKG_LIMIT={LIMIT}: SMOKE TEST on the first {LIMIT} queries per family; '
          f'files go to {RUN_DIR.name}/ and are not results')
print(f'MODE={MODE}  models={MODELS}  kgs={KGS}  families={FAMILIES}  seeds={SEEDS}  shuffles={SHUFFLES}')

MODE=real  models=['gpt-4.1-mini', 'gemini:gemini-3.5-flash-lite', 'llama3.3:70b']  kgs=['primekg', 'drkg', 'biokg']  families=['main', 'common']  seeds=[0, 1, 2]  shuffles=3


### Loading

In [3]:
# Free any KG state left over from a previous notebook, then the benchmark.
for _v in ('kgs', 'graphs', 'dossiers'):
    globals().pop(_v, None)
gc.collect()

ckpt = out.load_checkpoint('G01_benchmark')
bench = ckpt['benchmark']
assert ckpt['kgs'] == KGS and ckpt['pool_size'] == POOL_SIZE, 'G01 was built with different settings; rerun it'
pool = load_distractor_pool(BASE)
children = mondo_children(BASE)
print(f"{ckpt['n_pairs']} pairs ({ckpt['n_common']} common); source sha256 {ckpt['source_sha256']}")

128 pairs (43 common); source sha256 bbb9fb8748247517db73b9557235e97efc6572661321855c109267a1605755e2


In [4]:
# Models that cannot be called are dropped now, with the reason, rather than
# failing one query at a time inside the loop. A local model whose run files
# are already complete needs no server: the runner makes no calls for it, so
# the analysis stage can rerun this notebook on a node without Ollama.
def _runs_complete(m):
    """Every family x seed run file for m exists and carries no errored row."""
    for fam in FAMILIES:
        for seed in SEEDS:
            path = run_path(RUN_DIR, fam, m, seed)
            if not path.exists() or pd.read_csv(path, usecols=['error'])['error'].notna().any():
                return False
    return True

if not MOCK:
    try:
        pulled = ollama_models_available()
    except SystemExit as e:
        pulled = set()
        print(f'  {e}')
    usable = []
    for m in MODELS:
        if is_api_model(m):
            if api_key_for(m):
                usable.append(m)
            else:
                print(f'  no API key for {m}; skipping')
        elif m in pulled:
            usable.append(m)
        elif _runs_complete(m):
            usable.append(m)
            print(f'  {m}: not served here, but its run files are complete; reading them')
        else:
            print(f'  not pulled in Ollama and runs incomplete: {m}; skipping')
    MODELS = usable
    if not MODELS:
        raise SystemExit('No usable models: set an API key or pull an Ollama model.')


  Ollama not reachable at http://127.0.0.1:11434/api/tags: HTTPConnectionPool(host='127.0.0.1', port=11434): Max retries exceeded with url: /api/tags (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))
  not pulled in Ollama and runs incomplete: llama3.3:70b; skipping


In [5]:
# One dossier index per KG arm, built once. Loading is the expensive step; the
# per-query lookups afterwards are dictionary reads.
missing = [kg for kg in KGS if f'grounded_{kg}' not in bench.columns or f'shown_{kg}' not in bench.columns]
if missing:
    raise KeyError(f'the G01 checkpoint has no grounded or shown flag for {missing}; rebuild the pair status '
                   f'(tools/build_post_cutoff_approvals.py) if needed and rerun 01_benchmark first')
dossiers = {}
if not MOCK:
    entrez = load_entrez_to_symbol(BASE)
    for kg in KGS:
        print(f'loading {kg} ...')
        dossiers[kg] = KGDossiers(kg, config, BASE, cap=CAP, anonymize_genes=ANONYMIZE_GENES,
                                  bridge_mode=BRIDGE_MODE, entrez_to_symbol=entrez)
        n_drug = sum(dossiers[kg].resolves(d) for d in bench.drugbank_id)
        n_dis = sum(dossiers[kg].resolves_disease(m) for m in bench.mondo_id)
        print(f'  {kg}: resolves {n_drug}/{len(bench)} positives and {n_dis}/{len(bench)} diseases')

loading primekg ...


  primekg: resolves 108/128 positives and 122/128 diseases
loading drkg ...


  drkg: resolves 103/128 positives and 85/128 diseases
loading biokg ...


  biokg: resolves 100/128 positives and 79/128 diseases


## 1. Queries

Rebuilt from the seed, identically to G01.

In [6]:
queries = {(fam, seed): build_queries(bench, pool, family=fam, pool_size=POOL_SIZE, seed=seed, children=children)
           for fam in FAMILIES for seed in SEEDS}
if LIMIT:
    queries = {k: qs[:LIMIT] for k, qs in queries.items()}
for (fam, seed), qs in queries.items():
    print(f'{fam} seed {seed}: {len(qs)} queries')

main seed 0: 128 queries
main seed 1: 128 queries
main seed 2: 128 queries
common seed 0: 43 queries
common seed 1: 43 queries
common seed 2: 43 queries


## 2. Ranking loop

Idempotent: a (family, model, seed) file that already exists is loaded, not recomputed, unless `FORCE`. An arm that hits eight consecutive rate-limit errors is abandoned for that model and reported below, so a later launch with `BKG_FORCE=1` can complete it.

In [7]:
def _progress(total, desc):
    return tqdm(total=total, desc=desc, leave=False)

frames, aborted = [], []
for (fam, seed), qs in queries.items():
    df, ab = run_family(family=fam, queries=qs, models=MODELS, kgs=KGS, dossiers=dossiers, mock=MOCK,
                        seed=seed, shuffles=SHUFFLES, pool_size=POOL_SIZE, anonymize=ANONYMIZE,
                        temperature=TEMPERATURE, api_delay=API_DELAY, out_dir=RUN_DIR,
                        example_dir=EXAMPLE_DIR, force=FORCE, progress_factory=_progress)
    frames.append(df)
    aborted += [(fam, seed, m, arm) for m, arm in ab]
runs = pd.concat(frames, ignore_index=True)
print(f'{len(runs)} rows over {runs.model.nunique()} models, {runs.arm.nunique()} arms, '
      f'{runs.family.nunique()} families, {runs.seed.nunique()} seeds')
if aborted:
    print('incomplete (rate-limited) arms:', aborted)
err = runs[runs.error.notna()]
if len(err):
    # Errored rows are not scored: the next launch redoes them. Print them so
    # a run that is quietly losing calls is visible in the job log.
    print(f'{len(err)} errored rows will be redone on the next launch:')
    print(err.groupby(['family', 'model', 'arm']).size().to_string())
    print(err.error.value_counts().head(3).to_string())
# The benchmark's shown_<kg> flag (the graph holds the drug with a renderable
# gene or pathway neighbour) was computed from the graph before any call;
# pos_has_evidence records what the model was actually shown. Same rule, same
# graph, so they must agree row for row. grounded_<kg> is stricter (it also
# needs coverage: drug degree >= 5 and the disease in the graph) and is not
# the comparison here: a dossier is rendered whenever the graph has one.
if not MOCK:
    flags = bench.set_index('pair_id')
    kg_rows = runs[runs.arm != 'none']
    expected = np.array([bool(flags.at[p, f'shown_{a}']) for p, a in zip(kg_rows.pair_id, kg_rows.arm)])
    mismatch = kg_rows[kg_rows.pos_has_evidence.astype(bool).to_numpy() != expected]
    if len(mismatch):
        raise RuntimeError(f'{len(mismatch)} rows where pos_has_evidence disagrees with the benchmark shown flag; '
                           f'the status builder and the dossier resolve these drugs to different nodes:\n'
                           f'{mismatch[["arm", "pair_id", "drug", "pos_has_evidence"]].drop_duplicates().to_string()}')
    print('pos_has_evidence agrees with shown_<kg> on every KG-arm row')

12312 rows over 2 models, 4 arms, 2 families, 3 seeds
pos_has_evidence agrees with shown_<kg> on every KG-arm row


## 3. Run summary

MRR and hit rates per model, family and arm, pooled over queries, seeds and shuffles. `evidence_pos` is the share of KG-arm calls where the positive carried a dossier; `evidence_n` the mean number of candidates with one. A positive that is the only candidate with evidence is easy to spot for the wrong reason, which is why both are recorded.

In [8]:
summary = (runs.groupby(['family', 'model', 'arm'])
           .agg(n=('rank', 'size'), mrr=('reciprocal_rank', 'mean'), hit1=('hit1', 'mean'),
                hit3=('hit3', 'mean'), hit5=('hit5', 'mean'), parse_rate=('parsed', 'mean'),
                errors=('error', lambda s: int(s.notna().sum())),
                evidence_pos=('pos_has_evidence', 'mean'), evidence_n=('n_with_evidence', 'mean'),
                reliance=('frac_basis_kg', 'mean'), confidence=('mean_confidence', 'mean'))
           .reset_index())
summary['model_label'] = summary.model.map(LABELS).fillna(summary.model)
summary['arm_label'] = summary.arm.map(LABELS).fillna(summary.arm)
display(summary.set_index(['family', 'model_label', 'arm_label'])
        [['n', 'mrr', 'hit1', 'hit3', 'parse_rate', 'errors', 'evidence_pos', 'evidence_n', 'reliance']].round(3))
if LIMIT:
    print('smoke test: run_summary.csv not written')
else:
    # One summary per launch. A cluster launch covers one model and one seed
    # and several run at once, so a shared file would keep only the last
    # writer's rows; the analysis stage reads runs/, not these summaries.
    from biokgsuite.grounding.runner import model_slug
    scope = ('run_summary' if len(MODELS) > 1 or len(SEEDS) > 1
             else f'run_summary__{model_slug(MODELS[0])}__seed{SEEDS[0]}')
    out.table(summary, scope)

n    mrr   hit1   hit3  parse_rate  \
family model_label           arm_label                                          
common Gemini-3.5-Flash-Lite BioKG       387  0.809  0.705  0.894       1.000   
                             DRKG        387  0.828  0.734  0.904       1.000   
                             No KG       387  0.275  0.109  0.256       1.000   
                             PrimeKG     387  0.847  0.767  0.907       1.000   
       GPT-4.1-mini          BioKG       387  0.867  0.791  0.933       1.000   
                             DRKG        387  0.882  0.786  0.987       1.000   
                             No KG       387  0.352  0.134  0.408       1.000   
                             PrimeKG     387  0.875  0.788  0.964       0.997   
main   Gemini-3.5-Flash-Lite BioKG      1152  0.655  0.531  0.694       1.000   
                             DRKG       1152  0.613  0.489  0.640       1.000   
                             No KG      1152  0.284  0.115  0.276       1.000   
                             PrimeKG    1152  0.695  0.582  0.739       1.000   
       GPT-4.1-mini          BioKG      1152  0.633  0.525  0.641       0.994   
                             DRKG       1152  0.624  0.510  0.633       1.000   
                             No KG      1152  0.342  0.128  0.379       1.000   
                             PrimeKG    1152  0.703  0.602  0.728       0.997   

                                        errors  evidence_pos  evidence_n  \
family model_label           arm_label                                     
common Gemini-3.5-Flash-Lite BioKG           0         1.000       5.760   
                             DRKG            0         1.000       6.047   
                             No KG           0         0.000       0.000   
                             PrimeKG         0         1.000       5.566   
       GPT-4.1-mini          BioKG           0         1.000       5.760   
                             DRKG            0         1.000       6.047   
                             No KG           0         0.000       0.000   
                             PrimeKG         0         1.000       5.566   
main   Gemini-3.5-Flash-Lite BioKG           0         0.695       5.221   
                             DRKG            0         0.680       5.583   
                             No KG           0         0.000       0.000   
                             PrimeKG         0         0.766       5.026   
       GPT-4.1-mini          BioKG           0         0.695       5.221   
                             DRKG            0         0.680       5.583   
                             No KG           0         0.000       0.000   
                             PrimeKG         0         0.766       5.026   

                                        reliance  
family model_label           arm_label            
common Gemini-3.5-Flash-Lite BioKG         0.384  
                             DRKG          0.494  
                             No KG         0.079  
                             PrimeKG       0.456  
       GPT-4.1-mini          BioKG         0.609  
                             DRKG          0.696  
                             No KG         0.561  
                             PrimeKG       0.523  
main   Gemini-3.5-Flash-Lite BioKG         0.431  
                             DRKG          0.514  
                             No KG         0.059  
                             PrimeKG       0.392  
       GPT-4.1-mini          BioKG         0.605  
                             DRKG          0.659  
                             No KG         0.567  
                             PrimeKG       0.511

### Checkpoint

In [9]:
if LIMIT:
    print('smoke test: no checkpoint written')
else:
    out.checkpoint({
        'mode': MODE, 'models': MODELS, 'kgs': KGS, 'families': FAMILIES, 'seeds': SEEDS, 'shuffles': SHUFFLES,
        'pool_size': POOL_SIZE, 'cap': CAP, 'anonymize': ANONYMIZE, 'bridge_mode': BRIDGE_MODE,
        'run_files': sorted(p.name for p in RUN_DIR.glob('*.csv')),
        'aborted': aborted, 'summary': summary,
        'source_sha256': ckpt['source_sha256'],
    })


In [10]:
# Memory cleanup: free state before the next notebook in the pipeline.
for _v in ('kgs', 'graphs', 'dossiers', 'runs', 'bench'):
    globals().pop(_v, None)
gc.collect()
print('done:', out.notebook)

done: G02_run
